In [1]:
# For free tableau : https://getintopc.com/softwares/virtualization/tableau-desktop-professional-2021-free-download/

import pandas as pd
from json import load, dump

In [2]:
with open('../data/playlist_new.json') as pl:
    pl_videos = load(pl)['items']

In [3]:
playlist = pd.DataFrame.from_dict(pl_videos)
print(playlist.shape)
playlist.head()

(104, 4)


,videoId,videoPublishedAt,title,description
0,1z5-O7-5AXk,2022-11-07T16:53:03Z,Session 1 - Python Fundamentals | CampusX Data...,📃Chapters\n✒ Topic 0 - Introduction\n00:00:00 ...
1,JCkIrdrZEE8,2022-11-08T16:48:45Z,Session 2 - Operators + If-Else + Loops | Cam...,Session 2 - Operators + If-Else + Loops | Cam...
2,6HAu0Y9BjA4,2022-11-09T16:57:39Z,Session 3 - Python Strings | CampusX Data Sci...,Session 3 - Python Strings | CampusX Data Sci...
3,7ltjqU5iytY,2022-11-10T07:34:36Z,Programming Problems on Strings | Session 3 Su...,Programming Problems on Strings | Session 3 Su...
4,vg7BRC7wd0c,2022-11-10T17:01:20Z,How to Build a Portfolio Website for Data Science,Portfolio Website Examples-\nhttps://www.kunal...


In [4]:
playlist.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 104 entries, 0 to 103
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   videoId           104 non-null    object
 1   videoPublishedAt  103 non-null    object
 2   title             104 non-null    object
 3   description       103 non-null    object
dtypes: object(4)
memory usage: 3.4+ KB


### Categorize the videos as `isPaid, isSession, isSolutions`


In [5]:
playlist['isPaid'] = playlist['title'].apply(lambda x: 1 if 'Paid' in x else 0)


In [6]:
playlist['isSolutions'] = playlist['title'].apply(
    lambda x: 1 if 'Solutions' in x or 'Task' in x else 0)


In [7]:
playlist['isSession'] = playlist['title'].apply(
    lambda x: 1 if ('Session' in x or 'Paid' in x) and ('Solution' not in x) else 0)


### Give videos a module name.

In [8]:
def get_module_name(title: str) -> str:
    """ Does not works well. """
    modules = ['python', 'numpy', 'pandas',
               'seaborn', 'matplotlib', 'plotly', 'sql']

    for i in modules:
        if i in title.lower():
            return i.capitalize()
    else:
        return 'Others'


In [9]:
# playlist['moduleName'] = playlist['title'].apply(get_module_name)

#### This method does not works well. Need to approch another method.

Maybe I can bring the timeline of the modules. Then perform this task to get more precise result.


### Get the 'Task Number' and 'Session Number'.


In [23]:
playlist['sessionNo'] = playlist['title'].str.extract(r'Session (\d+)').astype(float)
playlist['taskNo'] = playlist['title'].str.extract(r'Task (\d+)').astype(float)

### Filtering `title` column.

#### Some rough works to filter the `title` column.

In [11]:
# rough_words = [
#     'Data Science Mentorship Program(DSMP) 2022-23',
#     'CampusX Data Science Mentorship Program',
#     'Guest Session',
#     'DSMP 2023',
#     'Free Session',
#     'Paid Recording',
#     'DSMP 2022-23',
#     'DSMP 2022 - 23',
#     'Paid Member Recording',
#     'Paid Recordings',
#     'Paid Recording',
#     'Friday Paid Session',
#     'Paid Members',
# ]

# playlist['roughTitle'] = (playlist['title']
#  .str.replace('|'.join(rough_words), '', regex=True)
#  .str.strip())


In [12]:
# playlist['roughTitle'].str.extract(r'Session \d{1,}? ?-? ?(.*) \|').isnull().sum()

In [13]:
# playlist['roughTitle_2'] = (playlist['roughTitle']
#                             .str.replace('Session on', 'Session - ', regex=False)
#                             .str.extract(r'- (.*) \|'))


#### Finally got the right method to filter the title column.

In [14]:
playlist['roughTitle_3'] = (playlist['title']
                            .str.split('|', regex=False)
                            .str.get(0)
                            .str.split('-')
                            .str.get(-1)
                            .str.replace('Session on', '')
                            .str.strip())


In [15]:
# Rename the final rough title column
playlist.rename(columns={'roughTitle_3': 'sessionName'}, inplace=True)

### Get data from `description` column.

In [16]:
desc = playlist['description']

In [17]:
playlist['linkInVideo'] = (playlist['description']
                           .str.extractall(r'(https?:\/\/[^ ]*)')
                           .unstack()
                           .applymap(lambda x: x.split('\n')[0] if isinstance(x, str) else x)
                           .apply(lambda x: [i for i in x if isinstance(i, str)], axis=1)
                           )

In [18]:
playlist['videoTimestamp'] = (playlist['description']
                              .str.extractall(r'(\b(?:\d+:)?\d+:\d+\b .*)')
                              .unstack()
                              .applymap(lambda x: x.split('\n')[0] if isinstance(x, str) else x)
                              .apply(lambda x: [i for i in x if isinstance(i, str)], axis=1)
                              )

### Export the **enhanced** dataset.

In [19]:
# Get the right form to export the dataset in json file
# Also fill nan values with `<NaN>` for convience.
# new_playlist = {'items': list(playlist
#                               .fillna('<NaN>')
#                               .drop(columns=['description', 'title'])
#                               .to_dict('index').values())}

In [20]:
# with open('../data/playlist_enhanced_new.json', 'w') as new_pl:
#     dump(new_playlist, new_pl, indent=2)